# Exploratory Data Analysis: Energy Demand

In this notebook I will explore an energy consumption dataset for Romania, fetched from [entsoe](https://www.entsoe.eu/data/power-stats/) using a purpose built script: [data_fetcher](../src/data_fetcher.py). During fetching, several columns were removed just before saving the csv file, like 'MeasureItem', 'CreateDate', 'UpdateDate', either due to inconsistent presence across yearly files in the 2019-2024 period, or they offered little analytical value.

Next I will load the dataset and explore some of its general properties using pandas.

In [157]:
import glob
import os

import pandas as pd

path = r'../data/raw'

```
all_files = glob.glob(os.path.join(path, '*.csv'))
df = pd.concat((pd.read_csv(f, parse_dates=['DateUTC', 'DateShort'], dayfirst=True) for f in all_files), ignore_index=True)
df = df.sort_values(by='DateUTC')
df.head()
```

Output:

```
DateUTC	                DateShort	TimeFrom     TimeTo	        CountryCode   Cov_ratio Value  Value_ScaleTo100
2019-01-01 00:00:00	2019-01-01	00:00	      01:00	          RO	     100	   5856.0	  5856.0
2019-01-01 01:00:00	2019-01-01	01:00	      02:00	          RO	     100	   5675.0	  5675.0
2019-01-01 02:00:00	2019-01-01	02:00	      03:00	          RO	     100	   5570.0	  5570.0
2019-01-01 03:00:00	2019-01-01	03:00	      04:00	          RO	     100	   5524.0	  5524.0
2019-01-01 04:00:00	2019-01-01	04:00	      05:00	          RO	     100	   5508.0	  5508.0
```

The next section examines the DateShort, TimeFrom and TimeTo columns. I expect these to be redundant, all derivable from the DateUTC column, with TimeTo inferable as DateUTC[i+1] - DateUTC[i]

```
diff_dates = df[df['DateUTC'].dt.date != df['DateShort']]
print(diff_dates[['DateUTC', 'DateShort']])

Empty DataFrame
Columns: [DateUTC, DateShort]
Index: []
```

The above confirms that DateShort is just the date portion extracted from DateUTC. Next I examine DateUTC vs TimeFrom:

```
tf = df[['DateUTC', 'TimeFrom', 'TimeTo']].copy()
tf['DateUTC'] = tf['DateUTC'].dt.strftime('%H:%M').str.strip()
tf['TimeFrom'] = tf['TimeFrom'].str[:5].str.strip()  # strip away the seconds
tf['TimeTo'] = tf['TimeTo'].str[:5].str.strip()  # strip away the seconds
tf['DateUTC_ref'] = df['DateUTC']

times_df = tf.loc[tf['DateUTC'] != tf['TimeFrom']]
print(len(times_df))
print(times_df)
```

Output:
```
5
      DateUTC TimeFrom TimeTo         DateUTC_ref
45876   03:00    02:00  03:00 2019-03-31 03:00:00
2114    03:00    02:00  03:00 2020-03-29 03:00:00
28267   03:00    02:00  03:00 2022-03-27 03:00:00
19482   03:00    02:00  03:00 2023-03-26 03:00:00
54661   03:00    02:00  03:00 2024-03-31 03:00:00
```

There 5 differences are all DST-related, spring transition. Column can be dropped. Finally DateUTC vs TimeTo:

```
delta_df = pd.to_timedelta(tf['TimeTo'] + ':00') - pd.to_timedelta(tf['TimeFrom'] + ':00')
delta_series = delta_df.dt.seconds / 3600 
hour_diff = delta_series[delta_series != 1.0]
print(len(hour_diff))
print(hour_diff)
```

Output:
```
0
Series([], dtype: float64)
```

Every row has exactly a 1h delta. TimeTo carries no information beyond TimeFrom + 1h. In conclusion, all three columns will be dropped in data_fetcher script.

```
all_files = glob.glob(os.path.join(path, '*.csv'))
df = pd.concat((pd.read_csv(f, parse_dates=['DateUTC'], dayfirst=True) for f in all_files), ignore_index=True)
df = df.sort_values(by='DateUTC')
df.head()
```

With redundant columns removed, the cleaned dataset looks as follows:

Output:
```
DateUTC	           Cov_ratio	Value	Value_ScaleTo100
01-01-2019 00:00	100	5856.0	5856.0
01-01-2019 01:00	100	5675.0	5675.0
01-01-2019 02:00	100	5570.0	5570.0
01-01-2019 03:00	100	5524.0	5524.0
01-01-2019 04:00	100	5508.0	5508.0
```

cov_ratio appears constant at 100. I will verify if this holds across all rows:

```
cov_ratio_diff = df[df['Cov_ratio'] != 100]
print(len(cov_ratio_diff))
```

Output:
```
0
```

cov_ratio is constant at 100 across all rows. It can be dropped.

Finally, I examine whether Value and Value_ScaleTo100 are identical:

```
val_diff = df[df['Value'] != df['Value_ScaleTo100']]
print(len(val_diff))
```

Output:
```
0
```

Value and Value_ScaleTo100 are identical across all rows, one can be dropped.

In [188]:
all_files = glob.glob(os.path.join(path, '*.csv'))
df = pd.concat((pd.read_csv(f, parse_dates=['DateUTC'], dayfirst=True) for f in all_files), ignore_index=True)
df = df.sort_values(by='DateUTC')
print(df.head())
print(f'Dataset size={len(df)}')
print(f'Contains NaN values: {df.isnull().values.any()}')

              DateUTC   Value
0 2019-01-01 00:00:00  5856.0
1 2019-01-01 01:00:00  5675.0
2 2019-01-01 02:00:00  5570.0
3 2019-01-01 03:00:00  5524.0
4 2019-01-01 04:00:00  5508.0
Dataset size=61282
Contains NaN values: False


The dataset has 2 columns and 61282 rows. Next I check for missing values.

```
time_delta = pd.to_datetime(df['DateUTC'].shift(-1), dayfirst=True) - pd.to_datetime(df['DateUTC'], dayfirst=True)
```

Output:
```
ValueError: time data "01/01/2021 00:00" doesn't match format "%d-%m-%Y %H:%M", at position 1439. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.
```

Before proceeding, I will fix the date format inconsistency inherited from the upstream source and re-download the dataset.

In [219]:
time_delta = pd.to_datetime(df['DateUTC'].shift(-1), dayfirst=True) - pd.to_datetime(df['DateUTC'], dayfirst=True)
days_diff_mask = time_delta.dt.days != 0
hour_diff_mask = time_delta.dt.seconds / 3600 != 1

days_diff = df[days_diff_mask]
hours_diff = pd.concat((df[hour_diff_mask], time_delta[hour_diff_mask]), axis=1)

print(f'Days difference: \n{len(days_diff)} \n{days_diff}')

print(f'Hours difference: \n{len(hours_diff)} \n{hours_diff}')

Days difference: 
1 
                  DateUTC    Value
60493 2025-12-31 23:00:00  5697.25
Hours difference: 
32 
                  DateUTC      Value         DateUTC
958   2019-02-09 22:00:00  6574.0000 0 days 02:00:00
2136  2019-03-31 01:00:00  5599.0000 0 days 02:00:00
2138  2019-03-31 03:00:00  5635.0000 0 days 00:00:00
10872 2020-03-29 01:00:00  4963.0000 0 days 02:00:00
10873 2020-03-29 03:00:00  4992.0000 0 days 00:00:00
18596 2021-02-13 21:00:00  7107.3925 0 days 02:00:00
18597 2021-02-13 23:00:00  6540.2500 0 days 03:00:00
18598 2021-02-14 02:00:00  6336.0000 0 days 02:00:00
18947 2021-02-28 16:00:00  1691.7500 0 days 15:00:00
19587 2021-03-27 21:00:00  4844.2500 0 days 12:00:00
19717 2021-04-02 19:00:00  7123.2500 0 days 02:00:00
21515 2021-06-16 18:00:00  7382.5000 0 days 11:00:00
21864 2021-07-01 17:00:00  5785.7500 0 days 04:00:00
22121 2021-07-12 13:00:00  3973.2500 0 days 08:00:00
22701 2021-08-06 00:00:00  1448.7500 0 days 05:00:00
22704 2021-08-06 07:00:00  5539.5000 0

December 31st 2025 is the last row of the dataset and can be ignored. 5 differences are due to DST. The March 31st, 2024 entry, there is a April 5th row misplaced row in the csv. There is genuinely missing data, i.e. February 9th, where the 23:00 hour is absent (the series jumps directly from 22:00 to 00:00)

DST discrepancies will be handled at processing stage, before data reaches the model. Small gaps, 2 to 3 hours, will be interpolated using similar days. Larger gaps, +8h, will handled differently. To prevent the model seeing discontinuities, the data will be split into continuous segments, sliding windows will be generated per segment, and all windows will then be merged into a single training set. The segment based approach handles the gaps naturally.

| DST | small/low peak gaps | large/high peak gaps |
|----------|----------|----------|
| 2019-03-31 01:00:00    | 2019-02-09 22:00:00 - 2h    | 2021-02-28 16:00:00 - 15h    |
| 2020-03-29 01:00:00    | 2021-02-13 21:00:00 - 2h    | 2021-03-27 21:00:00 - 12h    |
| 2022-03-27 01:00:00    | 2021-02-13 23:00:00 - 3h    | 2021-06-16 18:00:00 - 11h    |
| 2023-03-26 01:00:00   | 2021-02-14 02:00:00 - 2h   | 2021-07-01 17:00:00 - 4h    |
| 2024-03-31 01:00:00   | 2021-04-02 19:00:00 - 2h    | 2021-08-06 - 24h   |
| 2025-03-30 02:00:00  | 2021-08-19 04:00:00 - 3h  | 2021-09-08 06:00:00 - 4h   |
|     | 2021-08-22 23:00:00 - 5h  |  2021-09-10 17:00:00 - 4h |
|     | 2021-09-13 09:00:00 - 2h  |  2025-10-28 06:00:00 - 3h  |
|     | 2022-10-30 00:00:00 - 2h  |    |
|     | 2025-10-26 00:00:00 - 2h |    |
